# Model Benchmark Investigation

This notebook complements `scripts/benchmark_models.py` with custom, thesis-oriented inspection workflows.

## A. Total Training Evaluation

### 1. Aggregate Metrics Comparison

In [1]:
%matplotlib inline
from __future__ import annotations

import json
import re
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

def find_repo_root(start: Path | None = None) -> Path:
    p = (start or Path.cwd()).resolve()
    for cand in [p, *p.parents]:
        if (cand / "src").exists() and (cand / "scripts").exists():
            return cand
    raise FileNotFoundError("Could not detect repo root (expected folders: src/, scripts/).")

REPO_ROOT = find_repo_root()
SRC_DIR = REPO_ROOT / "src"
if str(SRC_DIR) not in sys.path:
    sys.path.insert(0, str(SRC_DIR))

from energy_trading.evaluation.gate_synced_analysis import (
    target_policy,
    gate_time_dplus1_filter,
    build_tail_metrics,
    aggregate_tail_metrics,
    compute_quantile_coverage,
    plot_tail_calibration,
    plot_quantile_coverage_bars,
)


from energy_trading.visualization.style import apply_geo_style, THESIS_PALETTE
from energy_trading.utils.run_context import resolve_model_run_dirs

apply_geo_style()

# -------------------------------
# Benchmark run configuration
# -------------------------------
MODEL_KEY_XGB = "xgboost"
MODEL_KEY_TFT = "tft"
MODEL_KEY_LINEAR = "linear"

MODEL_RUN_DIRS = resolve_model_run_dirs(
    repo_root=REPO_ROOT,
    models=(MODEL_KEY_XGB, MODEL_KEY_TFT, MODEL_KEY_LINEAR),
)
RUN_DIR_XGB = MODEL_RUN_DIRS[MODEL_KEY_XGB]
RUN_DIR_TFT = MODEL_RUN_DIRS[MODEL_KEY_TFT]
RUN_DIR_LINEAR = MODEL_RUN_DIRS[MODEL_KEY_LINEAR]

BENCHMARK_NAME = f"benchmark_{RUN_DIR_XGB.name}_{RUN_DIR_TFT.name}_{RUN_DIR_LINEAR.name}"
SPLIT = "test"
LEAD = 1
WINDOW_DAYS = 5
WEEK_HOURS = 24 * 7
WEEK_REFERENCE_PRED_COL = "pred_afrr_activation_price_neg"

BENCHMARK_DIR = REPO_ROOT / "artifacts" / "benchmarks" / BENCHMARK_NAME
SUMMARY_CSV = BENCHMARK_DIR / "benchmark_summary.csv"

print("Resolved run dirs:")
print(f" - xgboost: {RUN_DIR_XGB.name}")
print(f" - tft: {RUN_DIR_TFT.name}")
print(f" - linear: {RUN_DIR_LINEAR.name}")

TRUTH_COL_BY_PRED = {
    "pred_da_price": "target_da_price",
    "pred_afrr_activation_price_pos": "target_afrr_activation_price_vwap_pos",
    "pred_afrr_activation_price_neg": "target_afrr_activation_price_vwap_neg",
    "pred_afrr_capacity_price_pos": "target_afrr_capacity_price_pos",
    "pred_afrr_capacity_price_neg": "target_afrr_capacity_price_neg",
    "pred_afrr_activation_rate_pos": "target_afrr_activation_rate_pos",
    "pred_afrr_activation_rate_neg": "target_afrr_activation_rate_neg",
}

def load_manifest(run_dir: Path) -> dict:
    mp = run_dir / "manifest.json"
    if not mp.exists():
        raise FileNotFoundError(f"Missing manifest: {mp}")
    return json.loads(mp.read_text())

def resolve_truth_path(run_dir: Path, manifest: dict) -> Path:
    sm = run_dir / "summary_metrics.json"
    if sm.exists():
        payload = json.loads(sm.read_text())
        gp = payload.get("ground_truth_path", "")
        if gp and Path(gp).exists():
            return Path(gp)
    mp = manifest.get("ground_truth", {}).get("default_path", "")
    if mp and Path(mp).exists():
        return Path(mp)
    return REPO_ROOT / "data" / "features" / "all_data_features.parquet"

def load_truth_df(path: Path) -> pd.DataFrame:
    truth = pd.read_parquet(path)
    truth["timestamp_utc"] = pd.to_datetime(truth["timestamp_utc"], utc=True, errors="coerce")
    missing = [c for c in TRUTH_COL_BY_PRED.values() if c not in truth.columns]
    if missing:
        raise KeyError(f"Missing truth columns in {path}: {sorted(set(missing))}")
    return truth

def _matches_model_key(path: Path, model_key: str) -> bool:
    if not model_key:
        return True
    mk = model_key.strip().lower()
    name = path.name.lower()
    if mk in {"xgb", "xgboost"}:
        return "xgboost" in name
    if mk == "linear":
        return "linear" in name
    if mk == "tft":
        # TFT exports in this repo are often unlabeled in filename
        # (e.g., da_target_da_price_...); exclude explicit non-TFT families.
        return ("xgboost" not in name) and ("linear" not in name)
    return mk in name

def _discover_long_prediction_file(run_dir: Path, split: str, pred_col: str, model_key: str) -> Path:
    pred_dir = run_dir / "predictions"
    if not pred_dir.exists():
        raise FileNotFoundError(f"Missing predictions dir: {pred_dir}")

    pats = [
        f"*{split}*{pred_col}*long*.parquet",
        f"*{pred_col}*long*{split}*.parquet",
        f"*{pred_col}*{split}*long*.parquet",
    ]
    candidates: list[Path] = []
    for pat in pats:
        candidates.extend(sorted(pred_dir.glob(pat)))

    if model_key:
        candidates = [c for c in candidates if _matches_model_key(c, model_key)]

    if not candidates:
        raise FileNotFoundError(
            f"No long prediction file for pred_col={pred_col}, split={split}, model_key={model_key} in {pred_dir}"
        )
    return candidates[0]

def load_long_df(run_dir: Path, pred_col: str, model_key: str) -> pd.DataFrame:
    p = _discover_long_prediction_file(run_dir, SPLIT, pred_col, model_key)
    df = pd.read_parquet(p)
    ts_col = next((c for c in ["target_time_utc", "timestamp_utc", "timestamp"] if c in df.columns), None)
    if ts_col is None:
        raise KeyError(f"Missing timestamp col in {p}. Found: {list(df.columns)}")
    df = df.copy()
    df["target_time_utc"] = pd.to_datetime(df[ts_col], utc=True, errors="coerce")
    if "lead_time_h" in df.columns:
        df["lead_time_h"] = pd.to_numeric(df["lead_time_h"], errors="coerce")
    else:
        df["lead_time_h"] = 1
    if "p50" not in df.columns and "predicted_value" in df.columns:
        df["p50"] = pd.to_numeric(df["predicted_value"], errors="coerce")
    return df

def _model_has_pred_col(run_dir: Path, pred_col: str, model_key: str) -> bool:
    try:
        _discover_long_prediction_file(run_dir, SPLIT, pred_col, model_key)
        return True
    except Exception:
        return False

def compute_metrics_by_lead_for_model(pred_col: str, run_dir: Path, model_key: str) -> pd.DataFrame:
    truth_col = TRUTH_COL_BY_PRED[pred_col]
    long_df = load_long_df(run_dir, pred_col, model_key)
    pred_col_name = "p50" if "p50" in long_df.columns else "predicted_value"
    d = long_df[["target_time_utc", "lead_time_h", pred_col_name]].copy().rename(columns={pred_col_name: "y_pred"})

    t = truth_df[["timestamp_utc", truth_col]].copy().rename(columns={"timestamp_utc": "target_time_utc", truth_col: "y_true"})
    d = d.merge(t, on="target_time_utc", how="left")

    truth_by_ts = t.dropna(subset=["target_time_utc"]).set_index("target_time_utc")["y_true"]
    d["y_naive_24h"] = truth_by_ts.reindex(d["target_time_utc"] - pd.Timedelta(hours=24)).to_numpy()

    out = []
    for lead, g in d.groupby("lead_time_h", dropna=True):
        yt = pd.to_numeric(g["y_true"], errors="coerce").to_numpy(dtype=float)
        yp = pd.to_numeric(g["y_pred"], errors="coerce").to_numpy(dtype=float)
        yn = pd.to_numeric(g["y_naive_24h"], errors="coerce").to_numpy(dtype=float)
        m = np.isfinite(yt) & np.isfinite(yp)
        mn = m & np.isfinite(yn)
        if not np.any(m):
            continue
        mae = float(np.mean(np.abs(yt[m] - yp[m])))
        skill = np.nan
        if np.any(mn):
            mae_naive = float(np.mean(np.abs(yt[mn] - yn[mn])))
            if np.isfinite(mae_naive) and mae_naive != 0:
                skill = float(1.0 - mae / mae_naive)
        out.append({"lead_time_h": float(lead), "mae": mae, "skill_score_mae": skill})
    return pd.DataFrame(out).sort_values("lead_time_h").reset_index(drop=True)

mx = load_manifest(RUN_DIR_XGB)
truth_path = resolve_truth_path(RUN_DIR_XGB, mx)
truth_df = load_truth_df(truth_path)

COMMON_PRED_COLS = [
    p for p in sorted(TRUTH_COL_BY_PRED)
    if _model_has_pred_col(RUN_DIR_XGB, p, MODEL_KEY_XGB)
    and _model_has_pred_col(RUN_DIR_TFT, p, MODEL_KEY_TFT)
    and _model_has_pred_col(RUN_DIR_LINEAR, p, MODEL_KEY_LINEAR)
]
if not COMMON_PRED_COLS:
    raise ValueError("No common prediction columns found for configured model keys in the selected run dirs.")

summary_df = pd.read_csv(SUMMARY_CSV) if SUMMARY_CSV.exists() else pd.DataFrame()
summary_df

def load_merged_for_target(pred_col: str, lead: int = 1) -> pd.DataFrame:
    """Backward-compatible helper used by Section 4 deep-dive.

    Returns columns: target_time_utc, y_true, xgb_p50, tft_p50, lin_p50.
    """
    if pred_col not in TRUTH_COL_BY_PRED:
        raise KeyError(f"Unknown pred_col={pred_col}. Available: {sorted(TRUTH_COL_BY_PRED)}")

    truth_col = TRUTH_COL_BY_PRED[pred_col]
    base = truth_df[["timestamp_utc", truth_col]].copy().rename(
        columns={"timestamp_utc": "target_time_utc", truth_col: "y_true"}
    )
    base["target_time_utc"] = pd.to_datetime(base["target_time_utc"], utc=True, errors="coerce")

    def _model_part(run_dir: Path, model_key: str, out_col: str) -> pd.DataFrame:
        m = load_long_df(run_dir, pred_col, model_key).copy()
        if "lead_time_h" in m.columns:
            m = m.loc[pd.to_numeric(m["lead_time_h"], errors="coerce") == float(lead)].copy()
        pred_col_name = "p50" if "p50" in m.columns else "predicted_value"
        if pred_col_name not in m.columns:
            raise KeyError(f"Missing p50/predicted_value for {pred_col} in {run_dir}")
        m = m[["target_time_utc", pred_col_name]].copy().rename(columns={pred_col_name: out_col})
        m["target_time_utc"] = pd.to_datetime(m["target_time_utc"], utc=True, errors="coerce")
        m[out_col] = pd.to_numeric(m[out_col], errors="coerce")
        m = m.sort_values("target_time_utc").drop_duplicates(subset=["target_time_utc"], keep="last")
        return m

    xgb = _model_part(RUN_DIR_XGB, MODEL_KEY_XGB, "xgb_p50")
    tft = _model_part(RUN_DIR_TFT, MODEL_KEY_TFT, "tft_p50")
    lin = _model_part(RUN_DIR_LINEAR, MODEL_KEY_LINEAR, "lin_p50")

    merged = base.merge(xgb, on="target_time_utc", how="inner")
    merged = merged.merge(tft, on="target_time_utc", how="inner")
    merged = merged.merge(lin, on="target_time_utc", how="inner")
    merged = merged.sort_values("target_time_utc").reset_index(drop=True)

    # Ensure numeric payload for downstream MAE plotting.
    for c in ["y_true", "xgb_p50", "tft_p50", "lin_p50"]:
        merged[c] = pd.to_numeric(merged[c], errors="coerce")
    return merged


def _week_slice_by_start(df: pd.DataFrame, start_utc: str | pd.Timestamp, hours: int = 24 * 7) -> pd.DataFrame:
    """Backward-compatible helper for section 4 weekly windows."""
    out = df.copy()
    out["target_time_utc"] = pd.to_datetime(out["target_time_utc"], utc=True, errors="coerce")
    start = pd.to_datetime(start_utc, utc=True, errors="coerce")
    if pd.isna(start):
        return out.iloc[0:0].copy()
    end = start + pd.Timedelta(hours=int(hours))
    return out[(out["target_time_utc"] >= start) & (out["target_time_utc"] < end)].copy().reset_index(drop=True)


# Backward-compatible defaults for Section 4 deep-dive windows.
def _infer_default_week_starts_from_truth(
    *,
    truth_col: str = "target_afrr_activation_price_vwap_neg",
    hours: int = 24 * 7,
) -> tuple[pd.Timestamp, pd.Timestamp]:
    tdf = truth_df[["timestamp_utc", truth_col]].copy() if truth_col in truth_df.columns else truth_df[["timestamp_utc"]].copy()
    tdf["timestamp_utc"] = pd.to_datetime(tdf["timestamp_utc"], utc=True, errors="coerce")
    if truth_col in tdf.columns:
        tdf["y"] = pd.to_numeric(tdf[truth_col], errors="coerce")
    else:
        tdf["y"] = np.nan
    tdf = tdf.dropna(subset=["timestamp_utc"]).sort_values("timestamp_utc").reset_index(drop=True)
    if tdf.empty:
        now = pd.Timestamp.utcnow().tz_localize("UTC")
        base = now.floor("D")
        return (base - pd.Timedelta(days=14), base - pd.Timedelta(days=7))

    # Typical week: latest full week available.
    last_ts = tdf["timestamp_utc"].max()
    typical_start = (last_ts - pd.Timedelta(hours=hours)).floor("D")

    # High-volatility week: rolling std peak window.
    if tdf["y"].notna().sum() >= max(24, hours // 2):
        roll = tdf["y"].rolling(window=hours, min_periods=max(24, hours // 2)).std()
        if roll.notna().any():
            end_idx = int(roll.idxmax())
            high_start = tdf.loc[max(0, end_idx - hours + 1), "timestamp_utc"].floor("D")
        else:
            high_start = typical_start - pd.Timedelta(days=7)
    else:
        high_start = typical_start - pd.Timedelta(days=7)

    if high_start >= typical_start:
        high_start = typical_start - pd.Timedelta(days=7)
    return (pd.to_datetime(high_start, utc=True), pd.to_datetime(typical_start, utc=True))


if "HIGH_WEEK_START_UTC" not in globals() or "TYPICAL_WEEK_START_UTC" not in globals():
    _hws, _tws = _infer_default_week_starts_from_truth()
    HIGH_WEEK_START_UTC = _hws
    TYPICAL_WEEK_START_UTC = _tws


Resolved run dirs:
 - xgboost: fulltrain_2026-04-25T17-30-55Z_xgboost
 - tft: fulltrain_2026-04-25T17-30-55Z_tft
 - linear: fulltrain_2026-04-25T17-30-55Z_linear


### 2. Horizon Degradation (All Targets)

In [ ]:
import math

def _plot_horizon_series(ax, df: pd.DataFrame, x_col: str, y_col: str, *, label: str, color: str, linewidth: float = 2.0):
    x = pd.to_numeric(df[x_col], errors="coerce")
    y = pd.to_numeric(df[y_col], errors="coerce")
    m = x.notna() & y.notna()
    xs = x[m].to_numpy()
    ys = y[m].to_numpy()
    if len(xs) == 0:
        return
    if len(xs) == 1:
        # Linear baseline often has only lead=1; show explicit marker so it is visible.
        ax.plot(xs, ys, label=label, color=color, linewidth=linewidth, marker="o", markersize=6)
    else:
        ax.plot(xs, ys, label=label, color=color, linewidth=linewidth)

fig, axes = plt.subplots(len(COMMON_PRED_COLS), 2, figsize=(14, 4 * len(COMMON_PRED_COLS)), squeeze=False)

for r, pred_col in enumerate(COMMON_PRED_COLS):
    xgb_lead = compute_metrics_by_lead_for_model(pred_col, RUN_DIR_XGB, MODEL_KEY_XGB)
    tft_lead = compute_metrics_by_lead_for_model(pred_col, RUN_DIR_TFT, MODEL_KEY_TFT)
    lin_lead = compute_metrics_by_lead_for_model(pred_col, RUN_DIR_LINEAR, MODEL_KEY_LINEAR)

    if xgb_lead.empty or tft_lead.empty or lin_lead.empty:
        axes[r, 0].text(0.5, 0.5, f"missing lead metrics for {pred_col}", ha="center")
        axes[r, 1].text(0.5, 0.5, f"missing lead metrics for {pred_col}", ha="center")
        continue

    ax0 = axes[r, 0]
    _plot_horizon_series(ax0, xgb_lead, "lead_time_h", "mae", label="XGBoost", color=THESIS_PALETTE["primary"])
    _plot_horizon_series(ax0, tft_lead, "lead_time_h", "mae", label="TFT", color=THESIS_PALETTE["tertiary"])
    _plot_horizon_series(ax0, lin_lead, "lead_time_h", "mae", label="Linear", color=THESIS_PALETTE["secondary"])
    ax0.set_title(f"MAE over Horizon | {pred_col}")
    ax0.set_xlabel("Lead Time [h]")
    ax0.set_ylabel("MAE")
    ax0.legend()

    ax1 = axes[r, 1]
    if "skill_score_mae" in xgb_lead.columns and "skill_score_mae" in tft_lead.columns and "skill_score_mae" in lin_lead.columns:
        _plot_horizon_series(ax1, xgb_lead, "lead_time_h", "skill_score_mae", label="XGBoost", color=THESIS_PALETTE["primary"])
        _plot_horizon_series(ax1, tft_lead, "lead_time_h", "skill_score_mae", label="TFT", color=THESIS_PALETTE["tertiary"])
        _plot_horizon_series(ax1, lin_lead, "lead_time_h", "skill_score_mae", label="Linear", color=THESIS_PALETTE["secondary"])
        ax1.axhline(0.0, color="#666", linestyle="--", linewidth=1)
    else:
        ax1.text(0.5, 0.5, "skill_score_mae missing", ha="center")
    ax1.set_title(f"Skill Score over Horizon | {pred_col}")
    ax1.set_xlabel("Lead Time [h]")
    ax1.set_ylabel("Skill Score")
    ax1.legend()

plt.tight_layout()

### 3. Weekly Truth vs Predictions (Mon-Sun, All Targets)

### 4. Event Window Deep Dive (Selected Target, Mon-Sun)

In [ ]:
# Optional focused deep-dive for one target (uses SAME global weeks as section 3)
SELECTED_PRED_COL = "pred_afrr_activation_price_neg"

merged = load_merged_for_target(SELECTED_PRED_COL, lead=LEAD)
high_week = _week_slice_by_start(merged, HIGH_WEEK_START_UTC)
typical_week = _week_slice_by_start(merged, TYPICAL_WEEK_START_UTC)

for label, view, start_utc in [
    ("High-Volatility", high_week, HIGH_WEEK_START_UTC),
    ("Typical", typical_week, TYPICAL_WEEK_START_UTC),
]:
    if view.empty:
        continue
    mae_xgb = (view["xgb_p50"] - view["y_true"]).abs().mean()
    mae_tft = (view["tft_p50"] - view["y_true"]).abs().mean()
    mae_lin = (view["lin_p50"] - view["y_true"]).abs().mean()

    plt.figure(figsize=(14, 4.8))
    plt.plot(view["target_time_utc"], view["y_true"], label="Truth", color=THESIS_PALETTE["neutral_dark"], linewidth=2.2)
    plt.plot(view["target_time_utc"], view["xgb_p50"], label=f"XGBoost P50 (MAE={mae_xgb:.3f})", color=THESIS_PALETTE["primary"], linewidth=1.8)
    plt.plot(view["target_time_utc"], view["tft_p50"], label=f"TFT P50 (MAE={mae_tft:.3f})", color=THESIS_PALETTE["tertiary"], linewidth=1.8)
    plt.plot(view["target_time_utc"], view["lin_p50"], label=f"Linear P50 (MAE={mae_lin:.3f})", color=THESIS_PALETTE["secondary"], linewidth=1.8)
    plt.title(f"{label} Week Deep Dive | {SELECTED_PRED_COL} | lead={LEAD}")
    plt.xlabel("Timestamp (UTC)")
    plt.ylabel("Value")
    plt.xticks(rotation=20)
    plt.legend()
    plt.tight_layout()
    plt.show()

### 4. Residual & Bias Analysis

In [ ]:
# Build df for residual/regime analysis if not already provided
# Expected columns afterwards: target_time_utc, y_true, xgb_p50, tft_p50, lin_p50
ANALYSIS_PRED_COL = globals().get("SELECTED_PRED_COL", globals().get("WEEK_REFERENCE_PRED_COL", None))
if ANALYSIS_PRED_COL is None:
    ANALYSIS_PRED_COL = COMMON_PRED_COLS[0]

if "df" not in globals() or df is None:
    df = load_merged_for_target(ANALYSIS_PRED_COL, lead=LEAD)

required_cols = {"target_time_utc", "y_true", "xgb_p50", "tft_p50", "lin_p50"}
missing = required_cols - set(df.columns)
if missing:
    raise ValueError(
        f"df is missing required columns for Sections 4/5: {sorted(missing)}. "
        f"Current columns: {list(df.columns)}"
    )

print(f"Using ANALYSIS_PRED_COL={ANALYSIS_PRED_COL}, rows={len(df):,}")

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

# Expected columns in df: target_time_utc, y_true, xgb_p50, tft_p50, lin_p50
d = globals()["df"].copy()
d["target_time_utc"] = pd.to_datetime(d["target_time_utc"], utc=True, errors="coerce")
for c in ["y_true", "xgb_p50", "tft_p50", "lin_p50"]:
    d[c] = pd.to_numeric(d[c], errors="coerce")
d = d.dropna(subset=["target_time_utc", "y_true", "xgb_p50", "tft_p50", "lin_p50"]).sort_values("target_time_utc")

d["res_xgb"] = d["xgb_p50"] - d["y_true"]
d["res_tft"] = d["tft_p50"] - d["y_true"]
d["res_lin"] = d["lin_p50"] - d["y_true"]
xgb_color = THESIS_PALETTE["primary"]
tft_color = THESIS_PALETTE["tertiary"]
lin_color = THESIS_PALETTE["secondary"]

fig, axes = plt.subplots(2, 2, figsize=(18, 10), constrained_layout=True)

# 1) Predicted vs Actual
ax = axes[0, 0]
ax.scatter(d["y_true"], d["xgb_p50"], s=10, alpha=0.3, color=xgb_color, label="XGBoost")
ax.scatter(d["y_true"], d["tft_p50"], s=10, alpha=0.3, color=tft_color, label="TFT")
ax.scatter(d["y_true"], d["lin_p50"], s=10, alpha=0.3, color=lin_color, label="Linear")
vmin = float(np.nanmin([d["y_true"].min(), d["xgb_p50"].min(), d["tft_p50"].min(), d["lin_p50"].min()]))
vmax = float(np.nanmax([d["y_true"].max(), d["xgb_p50"].max(), d["tft_p50"].max(), d["lin_p50"].max()]))
ax.plot([vmin, vmax], [vmin, vmax], "k--", lw=1.5, label="Perfect prediction")
ax.set_xlabel("Actual (y_true)")
ax.set_ylabel("Predicted")
ax.set_title("Predicted vs Actual")
ax.legend(frameon=True)

# 2) Residual histogram XGB
ax = axes[0, 1]
sns.histplot(d["res_xgb"], bins=60, kde=True, color=xgb_color, ax=ax)
ax.axvline(0, color="black", linestyle="--", linewidth=1.2)
ax.set_xlabel("Residual (xgb_p50 - y_true)")
ax.set_ylabel("Count")
ax.set_title("Residuals: XGBoost")

# 3) Residual histogram TFT
ax = axes[1, 0]
sns.histplot(d["res_tft"], bins=60, kde=True, color=tft_color, ax=ax)
ax.axvline(0, color="black", linestyle="--", linewidth=1.2)
ax.set_xlabel("Residual (tft_p50 - y_true)")
ax.set_ylabel("Count")
ax.set_title("Residuals: TFT")

# 4) Residual histogram Linear
ax = axes[1, 1]
sns.histplot(d["res_lin"], bins=60, kde=True, color=lin_color, ax=ax)
ax.axvline(0, color="black", linestyle="--", linewidth=1.2)
ax.set_xlabel("Residual (lin_p50 - y_true)")
ax.set_ylabel("Count")
ax.set_title("Residuals: Linear")

plt.show()

### 5. Temporal Error Regimes

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

d = globals()["df"].copy()
d["target_time_utc"] = pd.to_datetime(d["target_time_utc"], utc=True, errors="coerce")
for c in ["y_true", "xgb_p50", "tft_p50", "lin_p50"]:
    d[c] = pd.to_numeric(d[c], errors="coerce")
d = d.dropna(subset=["target_time_utc", "y_true", "xgb_p50", "tft_p50", "lin_p50"]).sort_values("target_time_utc")

d["hour"] = d["target_time_utc"].dt.hour
d["dayofweek"] = d["target_time_utc"].dt.dayofweek  # Mon=0 ... Sun=6

d["ae_xgb"] = (d["xgb_p50"] - d["y_true"]).abs()
d["ae_tft"] = (d["tft_p50"] - d["y_true"]).abs()
d["ae_lin"] = (d["lin_p50"] - d["y_true"]).abs()

hour_index = list(range(24))
dow_index = list(range(7))
dow_labels = ["Mon", "Tue", "Wed", "Thu", "Fri", "Sat", "Sun"]

xgb_heat = (
    d.groupby(["hour", "dayofweek"], as_index=False)["ae_xgb"].mean()
    .pivot(index="hour", columns="dayofweek", values="ae_xgb")
    .reindex(index=hour_index, columns=dow_index)
)
tft_heat = (
    d.groupby(["hour", "dayofweek"], as_index=False)["ae_tft"].mean()
    .pivot(index="hour", columns="dayofweek", values="ae_tft")
    .reindex(index=hour_index, columns=dow_index)
)
lin_heat = (
    d.groupby(["hour", "dayofweek"], as_index=False)["ae_lin"].mean()
    .pivot(index="hour", columns="dayofweek", values="ae_lin")
    .reindex(index=hour_index, columns=dow_index)
)

xgb_heat.columns = dow_labels
tft_heat.columns = dow_labels
lin_heat.columns = dow_labels

vmax = float(np.nanmax([xgb_heat.to_numpy(), tft_heat.to_numpy(), lin_heat.to_numpy()]))
fig, axes = plt.subplots(1, 3, figsize=(26, 8), constrained_layout=True)

sns.heatmap(
    xgb_heat, ax=axes[0], cmap="YlOrRd", vmin=0, vmax=vmax,
    cbar=True, linewidths=0.2, linecolor="white"
)
axes[0].set_title("XGBoost MAE by Hour x DayOfWeek")
axes[0].set_xlabel("Day of Week")
axes[0].set_ylabel("Hour (UTC)")

sns.heatmap(
    tft_heat, ax=axes[1], cmap="YlOrRd", vmin=0, vmax=vmax,
    cbar=True, linewidths=0.2, linecolor="white"
)
axes[1].set_title("TFT MAE by Hour x DayOfWeek")
axes[1].set_xlabel("Day of Week")
axes[1].set_ylabel("Hour (UTC)")

sns.heatmap(
    lin_heat, ax=axes[2], cmap="YlOrRd", vmin=0, vmax=vmax,
    cbar=True, linewidths=0.2, linecolor="white"
)
axes[2].set_title("Linear MAE by Hour x DayOfWeek")
axes[2].set_xlabel("Day of Week")
axes[2].set_ylabel("Hour (UTC)")

plt.show()

delta_xgb_tft = xgb_heat - tft_heat
delta_xgb_lin = xgb_heat - lin_heat
delta_tft_lin = tft_heat - lin_heat

print("Delta Heatmap (XGB MAE - TFT MAE):")
display(delta_xgb_tft.round(4))
print("Delta Heatmap (XGB MAE - Linear MAE):")
display(delta_xgb_lin.round(4))
print("Delta Heatmap (TFT MAE - Linear MAE):")
display(delta_tft_lin.round(4))

### 6. Market-Specific Tail Performance Analysis

In [ ]:
from energy_trading.evaluation.gate_synced_analysis import build_tail_metrics, aggregate_tail_metrics, compute_quantile_coverage, plot_tail_calibration, plot_quantile_coverage_bars

# Tail analysis helpers + outputs (Market-Specific Tail Performance Analysis)
# DRY: uses reusable functions from energy_trading.evaluation.gate_synced_analysis

MODEL_INFO = {
    'xgb': {
        'label': 'XGBoost',
        'run_dir': RUN_DIR_XGB,
        'model_key': MODEL_KEY_XGB,
        'color': THESIS_PALETTE['primary'],
    },
    'tft': {
        'label': 'TFT',
        'run_dir': RUN_DIR_TFT,
        'model_key': MODEL_KEY_TFT,
        'color': THESIS_PALETTE['tertiary'],
    },
    'lin': {
        'label': 'Linear',
        'run_dir': RUN_DIR_LINEAR,
        'model_key': MODEL_KEY_LINEAR,
        'color': THESIS_PALETTE['secondary'],
    },
}
MODEL_IDS = ['xgb', 'tft', 'lin']
MODEL_LABELS = {mid: MODEL_INFO[mid]['label'] for mid in MODEL_IDS}
MODEL_COLORS = {mid: MODEL_INFO[mid]['color'] for mid in MODEL_IDS}


def _build_model_merged(pred_col: str) -> pd.DataFrame:
    truth_col = TRUTH_COL_BY_PRED[pred_col]
    base = truth_df[['timestamp_utc', truth_col]].copy().rename(
        columns={'timestamp_utc': 'target_time_utc', truth_col: 'y_true'}
    )
    base['target_time_utc'] = pd.to_datetime(base['target_time_utc'], utc=True, errors='coerce')
    merged = base

    for mid in MODEL_IDS:
        cfg = MODEL_INFO[mid]
        mdf = load_long_df(cfg['run_dir'], pred_col, cfg['model_key']).copy()
        if 'lead_time_h' in mdf.columns:
            mdf = mdf.loc[pd.to_numeric(mdf['lead_time_h'], errors='coerce') == float(LEAD)].copy()

        pred_col_name = 'p50' if 'p50' in mdf.columns else 'predicted_value'
        qcols = [q for q in ['p10','p20','p30','p40','p50','p60','p70','p80','p90'] if q in mdf.columns and q != pred_col_name]
        keep = ['snapshot_time_utc', 'target_time_utc', 'lead_time_h', pred_col_name, *qcols]
        mdf = mdf[keep].copy()

        rename = {pred_col_name: f'{mid}_p50'}
        for q in qcols:
            rename[q] = f'{mid}_{q}'
        mdf = mdf.rename(columns=rename)
        mdf = mdf.loc[:, ~mdf.columns.duplicated()].copy()

        if 'snapshot_time_utc' in merged.columns:
            mdf = mdf.drop(columns=[c for c in ['snapshot_time_utc', 'lead_time_h'] if c in mdf.columns], errors='ignore')
            merged = merged.merge(mdf, on='target_time_utc', how='inner')
        else:
            merged = merged.merge(mdf, on='target_time_utc', how='inner')

    merged = merged.loc[:, ~merged.columns.duplicated()].copy()
    merged['target_time_utc'] = pd.to_datetime(merged['target_time_utc'], utc=True, errors='coerce')
    if 'snapshot_time_utc' in merged.columns:
        merged['snapshot_time_utc'] = pd.to_datetime(merged['snapshot_time_utc'], utc=True, errors='coerce')
    return merged.sort_values('target_time_utc').reset_index(drop=True)


tail_parts = []
points_parts = []
cov_parts = []

for pred_col in COMMON_PRED_COLS:
    merged = _build_model_merged(pred_col)
    if merged.empty:
        continue

    tm, tp = build_tail_metrics(
        merged=merged,
        pred_col=pred_col,
        model_ids=MODEL_IDS,
        model_labels=MODEL_LABELS,
    )
    if not tm.empty:
        tail_parts.append(tm)
    if not tp.empty:
        points_parts.append(tp)

    cov = compute_quantile_coverage(
        merged,
        pred_col=pred_col,
        model_ids=MODEL_IDS,
        quantiles=(0.1, 0.9),
    )
    if not cov.empty:
        cov_parts.append(cov)


tail_metrics_df = pd.concat(tail_parts, ignore_index=True) if tail_parts else pd.DataFrame()
if tail_metrics_df.empty:
    raise ValueError('No tail metrics computed. Check COMMON_PRED_COLS / lead filtering / prediction files.')

print('Tail metrics rows:', len(tail_metrics_df))
display(tail_metrics_df.sort_values(['category', 'pred_col', 'segment', 'model']).reset_index(drop=True))

agg_tail = aggregate_tail_metrics(tail_metrics_df)
print('Aggregated tail metrics (category x segment x model):')
display(agg_tail)

agg_tail_pivot = agg_tail.pivot_table(
    index=['category', 'segment', 'model'],
    values=['segment_mae', 'normal_mae', 'tail_bias', 'spike_capture_rate'],
    aggfunc='first'
).reset_index()
print('Thesis-ready pivot table:')
display(agg_tail_pivot)

points_df = pd.concat(points_parts, ignore_index=True) if points_parts else pd.DataFrame()
plot_tail_calibration(
    points_df,
    model_colors=MODEL_COLORS,
    title_suffix='Global/Total Training Evaluation',
)

p90_calib_df = pd.concat(cov_parts, ignore_index=True) if cov_parts else pd.DataFrame()
print('p10/p90 calibration summary (all targets, lead filter only):')
display(p90_calib_df)

plot_quantile_coverage_bars(
    p90_calib_df,
    title_suffix='Global/Total Training Evaluation',
)


### 7. Final Decision Table (Global + Gate Metrics)

This section aggregates target-level model metrics into one decision table for simulation model selection.

Included per target/model:
- Global: MAE, RMSE, wMAPE, MBE, over-prediction ratio, directional accuracy
- Gate-closure metrics (when applicable): gate MAE / gate RMSE / gate rows
- Recommendation rule: prefer lowest gate MAE for gate-critical targets (DA, aFRR capacity), otherwise lowest global MAE (directional accuracy as tiebreak)


In [ ]:
from energy_trading.evaluation.metrics import compute_forecast_metrics, compute_gate_closure_metrics, gate_hour_for_target

MODEL_SPECS = [
    ("xgb", "XGBoost", RUN_DIR_XGB, MODEL_KEY_XGB, THESIS_PALETTE["primary"]),
    ("tft", "TFT", RUN_DIR_TFT, MODEL_KEY_TFT, THESIS_PALETTE["tertiary"]),
    ("lin", "Linear", RUN_DIR_LINEAR, MODEL_KEY_LINEAR, THESIS_PALETTE["secondary"]),
]


def _target_display_name(pred_col: str) -> str:
    p = pred_col.lower()
    if p == 'pred_da_price':
        return 'DA Price'
    if 'activation_rate' in p:
        return 'Activation Rate'
    if 'capacity_price' in p:
        return 'Capacity Price'
    if 'activation_price' in p:
        return 'Activation Price'
    return pred_col


def _model_long_lead_df(run_dir: Path, model_key: str, pred_col: str, lead: int) -> pd.DataFrame:
    d = load_long_df(run_dir, pred_col, model_key).copy()
    if 'lead_time_h' in d.columns:
        d = d.loc[pd.to_numeric(d['lead_time_h'], errors='coerce') == float(lead)].copy()
    return d


def _decision_sort_value(row: pd.Series) -> tuple:
    gate_mae = pd.to_numeric(pd.Series([row.get('gate_mae')]), errors='coerce').iloc[0]
    mae = pd.to_numeric(pd.Series([row.get('mae')]), errors='coerce').iloc[0]
    da = pd.to_numeric(pd.Series([row.get('directional_accuracy')]), errors='coerce').iloc[0]

    # Gate-critical targets: DA + capacity use gate MAE as primary sort key when available.
    is_gate_critical = bool(row.get('is_gate_critical', False))
    if is_gate_critical and pd.notna(gate_mae):
        primary = float(gate_mae)
    else:
        primary = float(mae) if pd.notna(mae) else float('inf')

    # Higher directional accuracy is better => use negative for ascending sort.
    secondary = float(-da) if pd.notna(da) else float('inf')
    tertiary = float(mae) if pd.notna(mae) else float('inf')
    return (primary, secondary, tertiary)


rows = []
for pred_col in COMMON_PRED_COLS:
    truth_col = TRUTH_COL_BY_PRED[pred_col]
    target_name = _target_display_name(pred_col)
    gate_h = gate_hour_for_target(truth_col)
    is_gate_critical = gate_h is not None

    for model_id, model_label, run_dir, model_key, _ in MODEL_SPECS:
        long_df = _model_long_lead_df(run_dir, model_key, pred_col, LEAD)
        if long_df.empty:
            rows.append({
                'pred_col': pred_col,
                'target': target_name,
                'truth_col': truth_col,
                'model': model_label,
                'model_id': model_id,
                'is_gate_critical': is_gate_critical,
                'gate_hour_local': gate_h,
                'error': 'empty long df at selected lead',
            })
            continue

        pred_value_col = 'p50' if 'p50' in long_df.columns else 'predicted_value'
        if pred_value_col not in long_df.columns:
            rows.append({
                'pred_col': pred_col,
                'target': target_name,
                'truth_col': truth_col,
                'model': model_label,
                'model_id': model_id,
                'is_gate_critical': is_gate_critical,
                'gate_hour_local': gate_h,
                'error': 'missing p50/predicted_value',
            })
            continue

        d_eval = long_df[['target_time_utc', pred_value_col]].copy().rename(columns={pred_value_col: 'y_pred'})
        t_eval = truth_df[['timestamp_utc', truth_col]].copy().rename(columns={'timestamp_utc': 'target_time_utc', truth_col: 'y_true'})
        merged = d_eval.merge(t_eval, on='target_time_utc', how='left')

        suite = compute_forecast_metrics(merged, y_true_col='y_true', y_pred_col='y_pred')

        gate_metrics = {}
        if gate_h is not None:
            gate_pred = long_df.copy()
            if 'predicted_value' not in gate_pred.columns and 'p50' in gate_pred.columns:
                gate_pred['predicted_value'] = pd.to_numeric(gate_pred['p50'], errors='coerce')
            gate_metrics = compute_gate_closure_metrics(
                pred_long_df=gate_pred,
                truth_df=truth_df,
                y_true_col=truth_col,
                y_pred_col='predicted_value',
                gate_hour_local=int(gate_h),
                timezone='Europe/Berlin',
                snapshot_col='snapshot_time_utc',
                target_time_col='target_time_utc',
            )

        rows.append({
            'pred_col': pred_col,
            'target': target_name,
            'truth_col': truth_col,
            'model': model_label,
            'model_id': model_id,
            'is_gate_critical': is_gate_critical,
            'gate_hour_local': gate_h,
            'n_rows_scored': suite.get('n_rows_scored'),
            'mae': suite.get('mae'),
            'rmse': suite.get('rmse'),
            'wmape': suite.get('wmape'),
            'mbe': suite.get('mbe'),
            'over_prediction_ratio': suite.get('over_prediction_ratio'),
            'directional_accuracy': suite.get('directional_accuracy'),
            'gate_n_rows': gate_metrics.get('n_rows_gate') if gate_metrics else np.nan,
            'gate_mae': gate_metrics.get('mae_gate') if gate_metrics else np.nan,
            'gate_rmse': gate_metrics.get('rmse_gate') if gate_metrics else np.nan,
            'error': '',
        })


decision_df = pd.DataFrame(rows)

if decision_df.empty:
    raise ValueError('Decision table is empty.')

# Rank and recommendation per target.
decision_df['sort_key'] = decision_df.apply(_decision_sort_value, axis=1)
decision_df = decision_df.sort_values(['pred_col', 'sort_key']).reset_index(drop=True)
decision_df['rank_within_target'] = decision_df.groupby('pred_col').cumcount() + 1
decision_df['recommended_for_simulation'] = decision_df['rank_within_target'] == 1

# Pretty display
disp_cols = [
    'target', 'pred_col', 'model', 'rank_within_target', 'recommended_for_simulation',
    'mae', 'rmse', 'wmape', 'mbe', 'over_prediction_ratio', 'directional_accuracy',
    'gate_hour_local', 'gate_n_rows', 'gate_mae', 'gate_rmse', 'error'
]

print('Final decision table (per-target ranking):')
display(decision_df[disp_cols].sort_values(['target', 'rank_within_target']).reset_index(drop=True))

best_models_df = decision_df[decision_df['recommended_for_simulation']].copy()
print('Recommended model per target:')
display(best_models_df[['target', 'pred_col', 'model', 'mae', 'gate_mae', 'directional_accuracy']].sort_values('target').reset_index(drop=True))

# Save artifacts for reporting
BENCHMARK_DIR.mkdir(parents=True, exist_ok=True)
out_csv = BENCHMARK_DIR / 'final_decision_table.csv'
out_best_csv = BENCHMARK_DIR / 'final_decision_recommended_models.csv'
decision_df.drop(columns=['sort_key']).to_csv(out_csv, index=False)
best_models_df.drop(columns=['sort_key']).to_csv(out_best_csv, index=False)
print('Wrote decision tables:')
print('-', out_csv)
print('-', out_best_csv)


### 8. Per-Target Verdict Table (Behavior + Trading Relevance)

This section unifies model-selection evidence into one concise per-target verdict table.

Inputs merged:
- Final per-target ranking from Section 7 (`decision_df`)
- Tail/spike behavior from Section 6 (`tail_metrics_df`)
- Activation-rate p90 calibration in high-imbalance periods (`p90_calib_df`)
- Horizon degradation proxy (MAE h_last / MAE h1)

Output artifacts:
- `final_per_target_verdict_table.csv`
- `final_per_target_verdict_table.json`


In [ ]:
# Build concise per-target verdict table
if 'decision_df' not in globals() or decision_df is None or decision_df.empty:
    raise RuntimeError('Run Section 7 first: decision_df not found or empty.')

if 'tail_metrics_df' not in globals() or tail_metrics_df is None or tail_metrics_df.empty:
    raise RuntimeError('Run Section 6 first: tail_metrics_df not found or empty.')

# Optional calibration table (only activation-rate targets + models with p90)
p90_df = globals().get('p90_calib_df', pd.DataFrame()).copy()
if p90_df is None:
    p90_df = pd.DataFrame()

model_id_to_label = {'xgb': 'XGBoost', 'tft': 'TFT', 'lin': 'Linear'}
label_to_model_id = {v: k for k, v in model_id_to_label.items()}

# Aggregate tail diagnostics to target/model level.
# Backward-compat: older cells used `n_tail`, new section 6 uses `n_segment`.
_n_tail_col = 'n_segment' if 'n_segment' in tail_metrics_df.columns else ('n_tail' if 'n_tail' in tail_metrics_df.columns else None)
if _n_tail_col is None:
    raise KeyError("tail_metrics_df must contain either 'n_segment' or 'n_tail'.")

tail_agg = (
    tail_metrics_df
    .groupby(['pred_col', 'model'], as_index=False)
    .agg(
        tail_mae_mean=('segment_mae', 'mean'),
        normal_mae_mean=('normal_mae', 'mean'),
        tail_bias_mean=('tail_bias', 'mean'),
        spike_capture_rate_mean=('spike_capture_rate', 'mean'),
        n_tail_total=(_n_tail_col, 'sum'),
    )
)
tail_agg['tail_mae_ratio'] = tail_agg['tail_mae_mean'] / tail_agg['normal_mae_mean']

# Horizon degradation proxy from per-model lead curves.
hrows = []
for pred_col in COMMON_PRED_COLS:
    for mid, mlabel, run_dir, model_key, _ in MODEL_SPECS:
        mdf = compute_metrics_by_lead_for_model(pred_col, run_dir, model_key)
        if mdf.empty:
            hrows.append({'pred_col': pred_col, 'model': mlabel, 'mae_h1': np.nan, 'mae_h_last': np.nan, 'mae_horizon_ratio': np.nan})
            continue
        mdf = mdf.sort_values('lead_time_h').reset_index(drop=True)
        mae_h1 = float(mdf.loc[mdf['lead_time_h'] == mdf['lead_time_h'].min(), 'mae'].iloc[0]) if not mdf.empty else np.nan
        mae_h_last = float(mdf['mae'].iloc[-1]) if not mdf.empty else np.nan
        ratio = (mae_h_last / mae_h1) if (np.isfinite(mae_h1) and mae_h1 > 1e-12 and np.isfinite(mae_h_last)) else np.nan
        hrows.append({'pred_col': pred_col, 'model': mlabel, 'mae_h1': mae_h1, 'mae_h_last': mae_h_last, 'mae_horizon_ratio': ratio})

horizon_agg = pd.DataFrame(hrows)

# Recommended model rows (one per target).
best = decision_df.loc[decision_df['recommended_for_simulation']].copy()
if best.empty:
    raise RuntimeError('No recommended rows in decision_df (expected one per target).')

verdict = best[['target', 'pred_col', 'truth_col', 'model', 'model_id', 'is_gate_critical', 'gate_hour_local', 'mae', 'rmse', 'wmape', 'mbe', 'over_prediction_ratio', 'directional_accuracy', 'gate_mae', 'gate_rmse', 'gate_n_rows']].copy()

verdict = verdict.merge(
    tail_agg[['pred_col', 'model', 'tail_mae_mean', 'normal_mae_mean', 'tail_mae_ratio', 'tail_bias_mean', 'spike_capture_rate_mean', 'n_tail_total']],
    on=['pred_col', 'model'],
    how='left',
)

verdict = verdict.merge(
    horizon_agg[['pred_col', 'model', 'mae_h1', 'mae_h_last', 'mae_horizon_ratio']],
    on=['pred_col', 'model'],
    how='left',
)

# Activation-rate p90 coverage lookup for chosen model.
if not p90_df.empty:
    p90_df = p90_df.copy()
    # Backward/forward compatibility: some tables store model as label, some as model_id.
    if 'model_id' not in p90_df.columns and 'model' in p90_df.columns:
        p90_df['model_id'] = p90_df['model'].map(label_to_model_id)
    if 'model' not in p90_df.columns and 'model_id' in p90_df.columns:
        p90_df['model'] = p90_df['model_id'].map(model_id_to_label)

    p90_df['model_id'] = p90_df.get('model_id', pd.Series(index=p90_df.index, dtype='object')).astype(str).str.lower()
    p90_df = p90_df[p90_df['model_id'].isin(model_id_to_label.keys())].copy()

    p90_df = p90_df.rename(columns={'p90_coverage_in_high_imbalance': 'activation_rate_p90_coverage_high_imbalance'})
    if 'activation_rate_p90_coverage_high_imbalance' in p90_df.columns:
        verdict = verdict.merge(
            p90_df[['pred_col', 'model_id', 'activation_rate_p90_coverage_high_imbalance']].drop_duplicates(),
            on=['pred_col', 'model_id'],
            how='left',
        )
    else:
        verdict['activation_rate_p90_coverage_high_imbalance'] = np.nan
else:
    verdict['activation_rate_p90_coverage_high_imbalance'] = np.nan


def _risk_band(row: pd.Series) -> str:
    flags = 0
    tmr = pd.to_numeric(pd.Series([row.get('tail_mae_ratio')]), errors='coerce').iloc[0]
    scr = pd.to_numeric(pd.Series([row.get('spike_capture_rate_mean')]), errors='coerce').iloc[0]
    hrr = pd.to_numeric(pd.Series([row.get('mae_horizon_ratio')]), errors='coerce').iloc[0]
    cov = pd.to_numeric(pd.Series([row.get('activation_rate_p90_coverage_high_imbalance')]), errors='coerce').iloc[0]

    if np.isfinite(tmr) and tmr > 1.8:
        flags += 1
    if np.isfinite(scr) and scr < 0.40:
        flags += 1
    if np.isfinite(hrr) and hrr > 1.6:
        flags += 1
    if 'activation_rate' in str(row.get('pred_col', '')) and np.isfinite(cov) and (cov < 0.80 or cov > 0.98):
        flags += 1

    if flags >= 2:
        return 'High'
    if flags == 1:
        return 'Medium'
    return 'Low'


def _key_assumption(row: pd.Series) -> str:
    if bool(row.get('is_gate_critical', False)):
        return 'Gate-closure execution quality is prioritized.'
    if 'activation_rate' in str(row.get('pred_col', '')):
        return 'Activation uncertainty is handled via quantile-aware safeguards.'
    return 'Global forecast error is representative for trading decisions.'


def _expected_behavior(row: pd.Series) -> str:
    tmr = pd.to_numeric(pd.Series([row.get('tail_mae_ratio')]), errors='coerce').iloc[0]
    hrr = pd.to_numeric(pd.Series([row.get('mae_horizon_ratio')]), errors='coerce').iloc[0]
    if np.isfinite(tmr) and tmr > 1.8:
        return 'Tail misses likely during stress windows; expect lower realized PnL in spikes.'
    if np.isfinite(hrr) and hrr > 1.6:
        return 'Forecast decays with horizon; expect weaker long-horizon dispatch quality.'
    return 'Stable behavior expected with moderate forecast-to-PnL translation risk.'


verdict['risk_level'] = verdict.apply(_risk_band, axis=1)
verdict['key_assumption'] = verdict.apply(_key_assumption, axis=1)
verdict['expected_simulation_behavior'] = verdict.apply(_expected_behavior, axis=1)

# Keep concise and ordered for thesis appendix.
order_cols = [
    'target', 'pred_col', 'model', 'risk_level',
    'mae', 'rmse', 'wmape', 'mbe', 'directional_accuracy',
    'gate_hour_local', 'gate_mae', 'gate_rmse',
    'tail_mae_mean', 'normal_mae_mean', 'tail_mae_ratio', 'tail_bias_mean', 'spike_capture_rate_mean',
    'mae_h1', 'mae_h_last', 'mae_horizon_ratio',
    'activation_rate_p90_coverage_high_imbalance',
    'key_assumption', 'expected_simulation_behavior',
]

verdict = verdict[order_cols].sort_values('target').reset_index(drop=True)

print('Per-target verdict table:')
display(verdict)

BENCHMARK_DIR.mkdir(parents=True, exist_ok=True)
out_csv = BENCHMARK_DIR / 'final_per_target_verdict_table.csv'
out_json = BENCHMARK_DIR / 'final_per_target_verdict_table.json'
verdict.to_csv(out_csv, index=False)
out_json.write_text(verdict.to_json(orient='records', indent=2), encoding='utf-8')
print('Wrote verdict artifacts:')
print('-', out_csv)
print('-', out_json)


### 9. Common Failure Flags (Automatic Pass/Warn/Fail)

This section computes automatic failure flags per target/model and stores them as benchmark artifacts.

Flag families:
- overfitting / instability (from training audit export when available)
- oversmoothing (tail underperformance + weak spike capture)
- systematic bias
- gate-closure failure (gate-critical targets)
- activation-rate quantile calibration failure

Outputs:
- `common_failure_flags.csv`
- `common_failure_flags.json`
- merged verdict with flags (`final_per_target_verdict_table_with_flags.*`)


In [ ]:
from pathlib import Path
import json
import numpy as np
import pandas as pd

FAIL_THRESH = {
    'tail_mae_ratio_max': 1.80,
    'spike_capture_min': 0.40,
    'bias_abs_floor': 0.05,
    'bias_vs_mae_mult': 0.30,
    'gate_mae_rel_mult': 1.10,
    'p90_cov_min': 0.80,
    'p90_cov_max': 0.98,
}


def _map_training_target_to_pred_col(target_name: str) -> str | None:
    t = str(target_name).lower()
    if 'target_da_price' in t:
        return 'pred_da_price'
    if 'target_afrr_activation_price_vwap_pos' in t:
        return 'pred_afrr_activation_price_pos'
    if 'target_afrr_activation_price_vwap_neg' in t:
        return 'pred_afrr_activation_price_neg'
    if 'target_afrr_capacity_price_pos' in t:
        return 'pred_afrr_capacity_price_pos'
    if 'target_afrr_capacity_price_neg' in t:
        return 'pred_afrr_capacity_price_neg'
    if 'target_afrr_activation_rate_pos' in t:
        return 'pred_afrr_activation_rate_pos'
    if 'target_afrr_activation_rate_neg' in t:
        return 'pred_afrr_activation_rate_neg'
    return None


def _load_training_audit_status() -> pd.DataFrame:
    p = REPO_ROOT / 'data/reports/model_audit/training_performance_audit_all_models.json'
    if not p.exists():
        return pd.DataFrame(columns=['model', 'pred_col', 'training_status'])
    try:
        payload = json.loads(p.read_text(encoding='utf-8'))
    except Exception:
        return pd.DataFrame(columns=['model', 'pred_col', 'training_status'])

    rows = payload.get('targets', []) if isinstance(payload, dict) else []
    out_rows = []
    for r in rows:
        model = str(r.get('model', '')).strip().lower()
        target = str(r.get('target', '')).strip()
        status = str(r.get('status', '')).strip()
        pred_col = _map_training_target_to_pred_col(target)
        if not model or not pred_col:
            continue
        if model in {'xgb'}:
            model = 'xgboost'
        out_rows.append({'model': model, 'pred_col': pred_col, 'training_status': status})

    if not out_rows:
        return pd.DataFrame(columns=['model', 'pred_col', 'training_status'])
    return pd.DataFrame(out_rows).drop_duplicates(subset=['model', 'pred_col'], keep='first')


if 'decision_df' not in globals() or decision_df is None or decision_df.empty:
    raise RuntimeError('Run Section 7 first: decision_df is required.')
if 'tail_metrics_df' not in globals() or tail_metrics_df is None or tail_metrics_df.empty:
    raise RuntimeError('Run Section 6 first: tail_metrics_df is required.')

base = decision_df.copy()
base['model'] = base['model'].astype(str)
base['model_id'] = base['model_id'].astype(str)

# If decision_df is unexpectedly huge, cap to one row per target/model rank for flag artifact.
if {'target', 'model', 'rank_within_target'}.issubset(base.columns):
    base = base.sort_values(['target', 'model', 'rank_within_target']).drop_duplicates(['target', 'model'], keep='first')

# Tail behavior features.
tail_model = (
    tail_metrics_df
    .groupby(['pred_col', 'model'], as_index=False)
    .agg(
        tail_mae_mean=('segment_mae', 'mean'),
        normal_mae_mean=('normal_mae', 'mean'),
        spike_capture_rate_mean=('spike_capture_rate', 'mean'),
        tail_bias_mean=('tail_bias', 'mean'),
    )
)
tail_model['tail_mae_ratio'] = tail_model['tail_mae_mean'] / tail_model['normal_mae_mean']

flags = base.merge(tail_model, on=['pred_col', 'model'], how='left')

# Training audit status (optional).
train_status_df = _load_training_audit_status()
if not train_status_df.empty:
    model_label_to_norm = {'XGBoost': 'xgboost', 'TFT': 'tft', 'Linear': 'linear'}
    flags['model_norm'] = flags['model'].map(model_label_to_norm)
    flags = flags.merge(
        train_status_df,
        left_on=['model_norm', 'pred_col'],
        right_on=['model', 'pred_col'],
        how='left',
        suffixes=('', '_audit'),
    )
    flags['training_status'] = flags['training_status'].fillna('')
else:
    flags['training_status'] = ''

# Gate baseline per target.
gate_baseline = (
    flags.groupby('pred_col', as_index=False)['gate_mae']
    .median(numeric_only=True)
    .rename(columns={'gate_mae': 'gate_mae_median'})
)
flags = flags.merge(gate_baseline, on='pred_col', how='left')

# Activation-rate p90 coverage (optional).
p90_df = globals().get('p90_calib_df', pd.DataFrame())
if p90_df is None or p90_df.empty:
    flags['activation_rate_p90_coverage_high_imbalance'] = np.nan
else:
    model_label_to_norm = {'XGBoost': 'xgboost', 'TFT': 'tft', 'Linear': 'linear'}
    p90_small = p90_df.copy()
    if 'model_norm' not in p90_small.columns:
        if 'model_id' in p90_small.columns:
            p90_small['model_norm'] = p90_small['model_id'].astype(str).str.lower()
        elif 'model' in p90_small.columns:
            p90_small['model_norm'] = p90_small['model'].map(model_label_to_norm)
        else:
            p90_small['model_norm'] = np.nan

    keep_cols = ['pred_col', 'model_norm', 'p90_coverage_in_high_imbalance']
    if all(c in p90_small.columns for c in keep_cols):
        flags = flags.merge(
            p90_small[keep_cols].drop_duplicates(),
            on=['pred_col', 'model_norm'],
            how='left',
        ).rename(columns={'p90_coverage_in_high_imbalance': 'activation_rate_p90_coverage_high_imbalance'})
    else:
        flags['activation_rate_p90_coverage_high_imbalance'] = np.nan

# Automatic flags.
flags['overfitting_flag'] = flags['training_status'].eq('Overfitting Detected')
flags['instability_flag'] = flags['training_status'].eq('Unstable')

flags['oversmoothing_flag'] = (
    (pd.to_numeric(flags['tail_mae_ratio'], errors='coerce') > FAIL_THRESH['tail_mae_ratio_max'])
    & (
        pd.to_numeric(flags['spike_capture_rate_mean'], errors='coerce').isna()
        | (pd.to_numeric(flags['spike_capture_rate_mean'], errors='coerce') < FAIL_THRESH['spike_capture_min'])
    )
)

abs_mbe = pd.to_numeric(flags['mbe'], errors='coerce').abs()
mae = pd.to_numeric(flags['mae'], errors='coerce')
bias_threshold = np.maximum(FAIL_THRESH['bias_abs_floor'], FAIL_THRESH['bias_vs_mae_mult'] * mae)
flags['bias_flag'] = abs_mbe > bias_threshold

flags['gate_failure_flag'] = (
    flags['is_gate_critical'].fillna(False).astype(bool)
    & pd.to_numeric(flags['gate_mae'], errors='coerce').notna()
    & pd.to_numeric(flags['gate_mae_median'], errors='coerce').notna()
    & (pd.to_numeric(flags['gate_mae'], errors='coerce') > FAIL_THRESH['gate_mae_rel_mult'] * pd.to_numeric(flags['gate_mae_median'], errors='coerce'))
)

cov = pd.to_numeric(flags['activation_rate_p90_coverage_high_imbalance'], errors='coerce')
is_act_rate = flags['pred_col'].astype(str).str.contains('activation_rate', case=False, regex=False)
flags['calibration_failure_flag'] = (
    is_act_rate
    & cov.notna()
    & ((cov < FAIL_THRESH['p90_cov_min']) | (cov > FAIL_THRESH['p90_cov_max']))
)

flag_cols = [
    'overfitting_flag',
    'instability_flag',
    'oversmoothing_flag',
    'bias_flag',
    'gate_failure_flag',
    'calibration_failure_flag',
]

flags['n_fail_flags'] = flags[flag_cols].fillna(False).sum(axis=1)
flags['failure_status'] = np.where(flags['n_fail_flags'] >= 2, 'FAIL', np.where(flags['n_fail_flags'] == 1, 'WARN', 'PASS'))

# Vectorized notes to avoid expensive axis=1 apply on large tables.
notes_parts = []
for c in flag_cols:
    label = c.replace('_flag', '')
    notes_parts.append(np.where(flags[c].fillna(False), label, ''))
notes_df = pd.DataFrame({c: np.array(v, dtype=object) for c, v in zip(flag_cols, notes_parts)})
flags['failure_notes'] = notes_df.apply(lambda r: ', '.join([x for x in r if x]), axis=1)

out_cols = [
    'target', 'pred_col', 'truth_col', 'model', 'model_id', 'rank_within_target',
    'recommended_for_simulation', 'is_gate_critical', 'gate_hour_local',
    'mae', 'rmse', 'wmape', 'mbe', 'over_prediction_ratio', 'directional_accuracy',
    'gate_mae', 'gate_rmse', 'gate_n_rows',
    'tail_mae_mean', 'normal_mae_mean', 'tail_mae_ratio', 'spike_capture_rate_mean', 'tail_bias_mean',
    'activation_rate_p90_coverage_high_imbalance', 'training_status',
    *flag_cols, 'n_fail_flags', 'failure_status', 'failure_notes'
]
out_cols = [c for c in out_cols if c in flags.columns]
common_failure_flags = flags[out_cols].sort_values([c for c in ['target', 'rank_within_target', 'model'] if c in out_cols]).reset_index(drop=True)

print('Automatic common-failure flags:')
display(common_failure_flags)

BENCHMARK_DIR.mkdir(parents=True, exist_ok=True)
flags_csv = BENCHMARK_DIR / 'common_failure_flags.csv'
flags_json = BENCHMARK_DIR / 'common_failure_flags.json'
common_failure_flags.to_csv(flags_csv, index=False)
# Write JSON directly to file to avoid building huge in-memory string.
common_failure_flags.to_json(flags_json, orient='records', indent=2)

if 'verdict' in globals() and verdict is not None and not verdict.empty:
    verdict_with_flags = verdict.merge(
        common_failure_flags[['pred_col', 'model', *[c for c in flag_cols if c in common_failure_flags.columns], 'n_fail_flags', 'failure_status', 'failure_notes']],
        on=['pred_col', 'model'],
        how='left',
    )
else:
    verdict_with_flags = common_failure_flags.copy()

verdict_flags_csv = BENCHMARK_DIR / 'final_per_target_verdict_table_with_flags.csv'
verdict_flags_json = BENCHMARK_DIR / 'final_per_target_verdict_table_with_flags.json'
verdict_with_flags.to_csv(verdict_flags_csv, index=False)
verdict_with_flags.to_json(verdict_flags_json, orient='records', indent=2)

print('Wrote common-failure artifacts:')
print('-', flags_csv)
print('-', flags_json)
print('-', verdict_flags_csv)
print('-', verdict_flags_json)


## B. Simulation-Synced Evaluation

### 10. Gate-Time Synced Evaluation (Trading-Relevant)


In [ ]:
from energy_trading.evaluation.gate_synced_analysis import build_tail_metrics, aggregate_tail_metrics, compute_quantile_coverage, plot_tail_calibration, plot_quantile_coverage_bars
import gc

# Section B: Simulation-synced analytics using reusable functions (DRY)
# Memory-safe version: process target-by-target, avoid keeping large intermediate frames.

MODEL_IDS = ["xgb", "tft", "lin"]
MODEL_LABELS = {mid: MODEL_INFO[mid]["label"] for mid in MODEL_IDS}
MODEL_COLORS = {mid: MODEL_INFO[mid]["color"] for mid in MODEL_IDS}
MAX_TAIL_POINTS_PER_SLICE = 2500


def _build_model_merged(pred_col: str) -> pd.DataFrame:
    truth_col = TRUTH_COL_BY_PRED[pred_col]
    base = truth_df[["timestamp_utc", truth_col]].copy().rename(
        columns={"timestamp_utc": "target_time_utc", truth_col: "y_true"}
    )
    base["target_time_utc"] = pd.to_datetime(base["target_time_utc"], utc=True, errors="coerce")
    merged = base

    for mid in MODEL_IDS:
        cfg = MODEL_INFO[mid]
        mdf = load_long_df(cfg["run_dir"], pred_col, cfg["model_key"]).copy()
        keep = ["snapshot_time_utc", "target_time_utc", "lead_time_h"]
        pred_col_name = "p50" if "p50" in mdf.columns else "predicted_value"
        keep.append(pred_col_name)
        qcols = [q for q in ["p10", "p20", "p30", "p40", "p50", "p60", "p70", "p80", "p90"] if q in mdf.columns and q != pred_col_name]
        keep.extend(qcols)
        mdf = mdf[keep].copy()

        rename = {pred_col_name: f"{mid}_p50"}
        for q in qcols:
            rename[q] = f"{mid}_{q}"
        mdf = mdf.rename(columns=rename)
        mdf = mdf.loc[:, ~mdf.columns.duplicated()].copy()

        if "snapshot_time_utc" in merged.columns:
            mdf = mdf.drop(columns=[c for c in ["snapshot_time_utc", "lead_time_h"] if c in mdf.columns], errors="ignore")
            merged = merged.merge(mdf, on="target_time_utc", how="inner")
        else:
            merged = merged.merge(mdf, on="target_time_utc", how="inner")

    merged = merged.loc[:, ~merged.columns.duplicated()].copy()
    merged["target_time_utc"] = pd.to_datetime(merged["target_time_utc"], utc=True, errors="coerce")
    if "snapshot_time_utc" in merged.columns:
        merged["snapshot_time_utc"] = pd.to_datetime(merged["snapshot_time_utc"], utc=True, errors="coerce")
    return merged.sort_values("target_time_utc").reset_index(drop=True)


b_gate_rows = []
b_tail_rows = []
b_tail_points_small = []
b_cov_rows = []

for pred_col in COMMON_PRED_COLS:
    merged_all = _build_model_merged(pred_col)
    if merged_all.empty or "snapshot_time_utc" not in merged_all.columns:
        continue

    merged_gate = gate_time_dplus1_filter(
        merged_all,
        pred_col=pred_col,
        snapshot_col="snapshot_time_utc",
        target_col="target_time_utc",
    )
    del merged_all
    gc.collect()

    if merged_gate.empty:
        del merged_gate
        gc.collect()
        continue

    for mid in MODEL_IDS:
        p50_col = f"{mid}_p50"
        if p50_col not in merged_gate.columns:
            continue
        y = pd.to_numeric(merged_gate["y_true"], errors="coerce")
        yp = pd.to_numeric(merged_gate[p50_col], errors="coerce")
        m = y.notna() & yp.notna()
        if not bool(m.any()):
            continue
        err = yp[m] - y[m]
        b_gate_rows.append(
            {
                "pred_col": pred_col,
                "model": MODEL_LABELS[mid],
                "model_id": mid,
                "n": int(m.sum()),
                "mae": float(np.mean(np.abs(err))),
                "rmse": float(np.sqrt(np.mean(err ** 2))),
                "bias": float(np.mean(err)),
            }
        )

    tm, tp = build_tail_metrics(
        merged=merged_gate,
        pred_col=pred_col,
        model_ids=MODEL_IDS,
        model_labels=MODEL_LABELS,
    )
    if not tm.empty:
        b_tail_rows.append(tm)
    if not tp.empty:
        # Keep bounded sample for plotting only.
        tp_small = (
            tp.groupby(["category", "pred_col", "segment", "model_id"], as_index=False, group_keys=False)
            .apply(lambda d: d.sample(n=min(len(d), MAX_TAIL_POINTS_PER_SLICE), random_state=42))
            .reset_index(drop=True)
        )
        b_tail_points_small.append(tp_small)

    cov = compute_quantile_coverage(
        merged_gate,
        pred_col=pred_col,
        model_ids=MODEL_IDS,
        quantiles=(0.1, 0.9),
    )
    if not cov.empty:
        cov["model"] = cov["model_id"].map(MODEL_LABELS)
        b_cov_rows.append(cov)

    del merged_gate, tm, tp, cov
    gc.collect()


gate_synced_metrics_df = pd.DataFrame(b_gate_rows)
gate_synced_tail_df = pd.concat(b_tail_rows, ignore_index=True) if b_tail_rows else pd.DataFrame()
gate_synced_tail_points_df = pd.concat(b_tail_points_small, ignore_index=True) if b_tail_points_small else pd.DataFrame()
gate_synced_coverage_df = pd.concat(b_cov_rows, ignore_index=True) if b_cov_rows else pd.DataFrame()

gate_synced_tail_agg_df = aggregate_tail_metrics(gate_synced_tail_df) if not gate_synced_tail_df.empty else pd.DataFrame()

print("Gate-synced aggregate metrics:")
display(gate_synced_metrics_df.sort_values(["pred_col", "model_id"]).reset_index(drop=True))
print("Gate-synced tail metrics:")
display(gate_synced_tail_df.sort_values(["pred_col", "segment", "model_id"]).reset_index(drop=True))
print("Gate-synced p10/p90 coverage:")
display(gate_synced_coverage_df.sort_values(["pred_col", "model_id", "quantile"]).reset_index(drop=True))

plot_tail_calibration(
    gate_synced_tail_points_df,
    model_colors=MODEL_COLORS,
    title_suffix="Evaluated at Gate Closure (D+1 Delivery)",
)
plot_quantile_coverage_bars(
    gate_synced_coverage_df,
    model_colors=MODEL_COLORS,
    title_suffix="Evaluated at Gate Closure (D+1 Delivery)",
)

if not gate_synced_metrics_df.empty:
    gate_target = (
        gate_synced_metrics_df.groupby(["pred_col", "model", "model_id"], as_index=False)
        .agg(gate_mae_mean=("mae", "mean"), gate_rmse_mean=("rmse", "mean"), gate_bias_mean=("bias", "mean"), n_total=("n", "sum"))
    )
else:
    gate_target = pd.DataFrame(columns=["pred_col", "model", "model_id", "gate_mae_mean", "gate_rmse_mean", "gate_bias_mean", "n_total"])

# Export artifacts.
BENCHMARK_DIR.mkdir(parents=True, exist_ok=True)
gate_synced_metrics_df.to_csv(BENCHMARK_DIR / "gate_time_target_metrics_raw.csv", index=False)
gate_target.to_csv(BENCHMARK_DIR / "gate_time_target_mae.csv", index=False)
gate_synced_coverage_df.to_csv(BENCHMARK_DIR / "gate_time_quantile_coverage.csv", index=False)

print("Saved gate-time artifacts:")
print("-", BENCHMARK_DIR / "gate_time_target_metrics_raw.csv")
print("-", BENCHMARK_DIR / "gate_time_target_mae.csv")
print("-", BENCHMARK_DIR / "gate_time_quantile_coverage.csv")
